# grounded-rag-qa: pipeline walkthrough

Step-by-step view of what `grounded-rag ingest` and `grounded-rag ask` do: chunking, embedding, indexing, scored retrieval and grounded generation.

Run from the repo root environment (`uv sync`), with `GROQ_API_KEY` in `.env`.

In [ ]:
import os
from dataclasses import replace

from grounded_rag.config import SAMPLE_SOURCE, Settings

# in-memory Qdrant so the notebook doesn't lock the CLI's ./qdrant_data folder
settings = replace(Settings.from_env(), qdrant_path=":memory:")
settings

## 1. Load and chunk
Chunk length is counted in `cl100k_base` tokens; the recursive splitter falls back from paragraphs to lines to words so no chunk exceeds the budget.

In [ ]:
import tiktoken
from grounded_rag.ingest import ENCODING, load_source, split_documents

docs = load_source(SAMPLE_SOURCE)
chunks = split_documents(docs, settings.chunk_size, settings.chunk_overlap)

enc = tiktoken.get_encoding(ENCODING)
sizes = [len(enc.encode(c.page_content)) for c in chunks]
print(f"{len(chunks)} chunks, tokens per chunk: min {min(sizes)}, max {max(sizes)}")
print(chunks[5].page_content)

## 2. Embed and index in Qdrant

In [ ]:
from grounded_rag.store import build_embeddings, index_documents

embeddings = build_embeddings(settings)
print("embedding dimensions:", len(embeddings.embed_query("hello")))

store = index_documents(chunks, embeddings, settings)

## 3. Top-k always returns k results
Compare top cosine similarity for answerable, out-of-scope and junk questions.

In [ ]:
questions = [
    "How many casual leaves am I entitled to?",
    "Can I work remotely?",
    "What is the stock option policy?",
    "asdf banana rocket",
]
for q in questions:
    hits = store.similarity_search_with_score(q, k=3)
    print(f"{q:50} " + "  ".join(f"{s:.3f}" for _, s in hits))

## 4. Retriever with a similarity floor

In [ ]:
from grounded_rag.chain import build_retriever

retriever = build_retriever(store, settings)
for q in questions:
    print(f"{q:50} -> {len(retriever.invoke(q))} chunks pass {settings.score_threshold}")

## 5. Grounded generation
No chunks above the threshold means the fallback is returned without an LLM call.

In [ ]:
from grounded_rag.chain import build_chain, build_llm

chain = build_chain(retriever, build_llm(settings))
for q in questions:
    result = chain.invoke(q)
    print(f"Q: {q}
A: {result['answer']}
")